In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Install packages
!pip install -q deap

# All imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import random
import numpy as np
from deap import base, creator, tools, gp
import operator
import dill
import os
from torch.utils.data import Dataset, DataLoader
from collections import defaultdict
import json
from datetime import datetime

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
BATCH_SIZE = 32

# Paths
BASE_DIR = '/content/drive/MyDrive/EC_Project'
CHECKPOINT_DIR = f'{BASE_DIR}/checkpoints/anti_convergence_run'
DATA_DIR = "/content/data"
RESULTS_DIR = f'{BASE_DIR}/results'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Set seeds
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("="*70)
print("🔥 ANTI-CONVERGENCE EVOLUTION SYSTEM")
print("="*70)
print(f"✅ Device: {DEVICE}")
print(f"✅ Checkpoints: {CHECKPOINT_DIR}")
print("="*70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🔥 ANTI-CONVERGENCE EVOLUTION SYSTEM
✅ Device: cuda
✅ Checkpoints: /content/drive/MyDrive/EC_Project/checkpoints/anti_convergence_run


In [ ]:
# ============================================================================
# GP OPERATORS (Same as before)
# ============================================================================

def safe_sqrt(x):
    return torch.sqrt(torch.clamp(torch.abs(x), min=1e-7))

def safe_log(x):
    return torch.log(torch.clamp(torch.abs(x), min=1e-7))

def safe_tanh(x):
    return torch.tanh(x)

def safe_sigmoid(x):
    return torch.sigmoid(x)

def safe_softplus(x):
    return F.softplus(x)

def safe_elu(x):
    return F.elu(x)

# Create primitive set
pset = gp.PrimitiveSet("MAIN", 1)
pset.addPrimitive(torch.add, 2, name="add")
pset.addPrimitive(torch.mul, 2, name="mul")
pset.addPrimitive(torch.sub, 2, name="sub")
pset.addPrimitive(torch.maximum, 2, name="max")
pset.addPrimitive(torch.minimum, 2, name="min")
pset.addPrimitive(torch.abs, 1, name="abs")
pset.addPrimitive(torch.square, 1, name="square")
pset.addPrimitive(safe_sqrt, 1, name="sqrt")
pset.addPrimitive(safe_log, 1, name="log")
pset.addPrimitive(torch.neg, 1, name="neg")
pset.addPrimitive(torch.sin, 1, name="sin")
pset.addPrimitive(torch.cos, 1, name="cos")
pset.addPrimitive(safe_tanh, 1, name="tanh")
pset.addPrimitive(safe_sigmoid, 1, name="sigmoid")
pset.addPrimitive(safe_softplus, 1, name="softplus")
pset.addPrimitive(safe_elu, 1, name="elu")
pset.addEphemeralConstant("const", lambda: random.uniform(0, 2))
pset.renameArguments(ARG0='error')

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMax)

toolbox = base.Toolbox()
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=3)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("mate", gp.cxOnePoint)
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=5))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=5))

print("✅ GP system ready")

# ============================================================================
# DIVERSITY METRICS
# ============================================================================

def tree_to_string(tree):
    """Canonical string representation for comparison"""
    return str(tree)

def tree_distance(tree1, tree2):
    """
    Simple structural distance between trees.
    Returns 0 if identical, higher for more different.
    """
    s1 = tree_to_string(tree1)
    s2 = tree_to_string(tree2)

    if s1 == s2:
        return 0.0

    # Simple edit distance approximation
    len_diff = abs(len(s1) - len(s2))
    char_diff = sum(c1 != c2 for c1, c2 in zip(s1, s2))

    return (len_diff + char_diff) / max(len(s1), len(s2))

def is_trivial_tree(tree):
    """Check if tree is just a single operation on error"""
    s = str(tree)
    # Trivial patterns: square(error), abs(error), neg(error), etc.
    trivial_patterns = [
        "square(error)",
        "abs(error)",
        "neg(error)",
        "sqrt(error)",
        "log(error)",
        "sin(error)",
        "cos(error)",
        "tanh(error)",
        "sigmoid(error)",
        "exp(error)"
    ]
    return s in trivial_patterns

def get_tree_structure_hash(tree):
    """
    Get structural signature ignoring constants.
    Example: square(error) and square(mul(error, 1.5)) have different hashes
    """
    # Use string representation but normalize constants
    s = str(tree)
    # Remove specific constant values
    import re
    normalized = re.sub(r'\d+\.\d+', 'C', s)
    return normalized

print("✅ Diversity metrics ready")

✅ GP system ready
✅ Diversity metrics ready


/usr/local/lib/python3.12/dist-packages/deap/gp.py:257: RuntimeWarning: Ephemeral const function cannot be pickled because its generating function is a lambda function. Use functools.partial instead.
  warnings.warn("Ephemeral {name} function cannot be "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
/usr/local/lib/python3.12/dist-packages/deap/creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


In [ ]:
# ============================================================================
# EVOLVED LOSS FUNCTION
# ============================================================================

class EvolvedLossFunction(nn.Module):
    def __init__(self, tree):
        super().__init__()
        self.tree = tree
        self.func = gp.compile(expr=tree, pset=pset)

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs)
        one_hot.scatter_(1, targets.view(-1, 1), 1)
        error = one_hot - probs

        try:
            loss_val = self.func(error)
            if not torch.is_tensor(loss_val):
                loss_val = torch.tensor(loss_val, device=logits.device, dtype=torch.float32)
            if not loss_val.requires_grad:
                loss_val = loss_val + 0.0 * probs.sum()
            if loss_val.dim() > 1:
                loss_val = torch.mean(torch.sum(loss_val, dim=1))
            elif loss_val.dim() == 1:
                loss_val = torch.mean(loss_val)
            if torch.isnan(loss_val) or torch.isinf(loss_val):
                return 1000.0 * torch.mean(probs)
            return loss_val
        except:
            return 1000.0 * torch.mean(probs)

# ============================================================================
# MODEL
# ============================================================================

class OmniglotCNN(nn.Module):
    def __init__(self, num_classes=50):
        super(OmniglotCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# ============================================================================
# DATASET
# ============================================================================

from torchvision.datasets import Omniglot

transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

omniglot = Omniglot(root=DATA_DIR, background=True, download=True, transform=transform)

class FewShotOmniglot(Dataset):
    def __init__(self, base_dataset, shots=5, ways=50, for_train=True, seed=42):
        random.seed(seed)

        class_to_idx = {}
        for i in range(len(base_dataset)):
            _, label = base_dataset[i]
            if label not in class_to_idx:
                class_to_idx[label] = []
            class_to_idx[label].append(i)

        selected = random.sample(list(class_to_idx.keys()), ways)
        self.label_map = {old: new for new, old in enumerate(selected)}

        self.indices = []
        for old_label in selected:
            all_idx = class_to_idx[old_label]
            shuffled = random.sample(all_idx, len(all_idx))
            if for_train:
                self.indices.extend(shuffled[:shots])
            else:
                self.indices.extend(shuffled[shots:shots+15])

        self.base = base_dataset

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        img, old_label = self.base[self.indices[idx]]
        return img, self.label_map[old_label]

train_data = FewShotOmniglot(omniglot, shots=5, ways=50, for_train=True)
test_data = FewShotOmniglot(omniglot, shots=5, ways=50, for_train=False)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ Dataset: {len(train_data)} train, {len(test_data)} test")

100%|██████████| 9.46M/9.46M [00:00<00:00, 169MB/s]


✅ Dataset: 250 train, 750 test


In [ ]:
# ============================================================================
# EVALUATION FUNCTION
# ============================================================================

def evaluate_loss(tree, gen, ind):
    """Train and evaluate a loss function"""
    model = OmniglotCNN(num_classes=50).to(DEVICE)
    criterion = EvolvedLossFunction(tree)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(15):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if torch.isnan(loss) or torch.isinf(loss):
                return 0.0

            loss.backward()
            optimizer.step()

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

# ============================================================================
# ANTI-CONVERGENCE MECHANISMS
# ============================================================================

def apply_fitness_sharing(population, sigma=0.3):
    """
    Mechanism 1: Fitness Sharing
    Penalize individuals that are too similar to others.
    """
    for ind in population:
        # Count similar individuals
        niche_count = 0
        for other in population:
            dist = tree_distance(ind, other)
            if dist < sigma:  # Within niche radius
                niche_count += 1

        # Shared fitness = original fitness / niche count
        if niche_count > 0:
            ind.shared_fitness = ind.fitness.values[0] / niche_count
        else:
            ind.shared_fitness = ind.fitness.values[0]

def calculate_novelty_bonus(tree, population_archive):
    """
    Mechanism 2: Novelty Bonus
    Reward unique tree structures.
    """
    tree_hash = get_tree_structure_hash(tree)

    # Count how many times this structure has appeared
    structure_count = sum(1 for t in population_archive if get_tree_structure_hash(t) == tree_hash)

    # Novelty bonus: inversely proportional to frequency
    # Max bonus: 2%, decays as structure becomes common
    if structure_count == 0:
        return 2.0  # Brand new structure: +2%
    elif structure_count == 1:
        return 1.0  # Seen once: +1%
    elif structure_count == 2:
        return 0.5  # Seen twice: +0.5%
    else:
        return 0.0  # Common structure: no bonus

def apply_trivial_penalty(tree):
    """
    Mechanism 3: Op Blacklisting
    Penalize trivial single-op trees.
    """
    if is_trivial_tree(tree):
        return 0.95  # 5% penalty for trivial trees
    return 1.0  # No penalty

def adaptive_mutation_rate(generation, diversity_score):
    """
    Mechanism 4: Adaptive Mutation
    Increase mutation when diversity is low.
    """
    base_rate = 0.3

    # If diversity is low, crank up mutation
    if diversity_score < 0.2:
        return 0.5  # High exploration
    elif diversity_score < 0.4:
        return 0.4  # Medium-high
    else:
        return base_rate  # Normal

def calculate_population_diversity(population):
    """
    Calculate average pairwise distance in population.
    Returns 0-1, where 1 is maximally diverse.
    """
    if len(population) < 2:
        return 1.0

    distances = []
    for i, ind1 in enumerate(population):
        for ind2 in population[i+1:]:
            distances.append(tree_distance(ind1, ind2))

    return np.mean(distances) if distances else 0.0

print("✅ Anti-convergence mechanisms ready")
print("   1. Fitness sharing (penalize clones)")
print("   2. Novelty bonus (reward unique structures)")
print("   3. Trivial penalty (discourage single-ops)")
print("   4. Adaptive mutation (increase when diversity drops)")

✅ Anti-convergence mechanisms ready
   1. Fitness sharing (penalize clones)
   2. Novelty bonus (reward unique structures)
   3. Trivial penalty (discourage single-ops)
   4. Adaptive mutation (increase when diversity drops)


In [ ]:
# ============================================================================
# CHECKPOINTING SYSTEM
# ============================================================================

def save_checkpoint(population, generation, metadata):
    """Save population and metadata to Drive using dill"""
    checkpoint = {
        'generation': generation,
        'population': population,
        'metadata': metadata,
        'timestamp': datetime.now().isoformat()
    }

    filepath = f'{CHECKPOINT_DIR}/checkpoint_gen_{generation}.pkl'
    with open(filepath, 'wb') as f:
        dill.dump(checkpoint, f)

    # Note: Removed JSON metadata export as gp.Primitive objects are not JSON-serializable.
    # Full state is preserved in the .pkl file above.
    print(f"💾 Saved checkpoint: gen {generation}")

def load_checkpoint(generation):
    """Load population from checkpoint"""
    filepath = f'{CHECKPOINT_DIR}/checkpoint_gen_{generation}.pkl'
    try:
        with open(filepath, 'rb') as f:
            checkpoint = dill.load(f)
        print(f"📂 Loaded checkpoint: gen {generation}")
        return checkpoint
    except FileNotFoundError:
        print(f"⚠️  No checkpoint found for gen {generation}")
        return None

def find_latest_checkpoint():
    """Find the most recent checkpoint"""
    checkpoints = [f for f in os.listdir(CHECKPOINT_DIR) if f.startswith('checkpoint_gen_')]
    if not checkpoints:
        return None

    # Extract generation numbers
    gen_nums = [int(f.split('_')[-1].replace('.pkl', '')) for f in checkpoints]
    latest_gen = max(gen_nums)

    return load_checkpoint(latest_gen)

# ============================================================================
# MAIN EVOLUTION LOOP
# ============================================================================

def run_anti_convergence_evolution(
    pop_size=20,
    n_generations=20,
    resume=True
):
    """
    Main evolution with all anti-convergence mechanisms.
    """
    print("\n" + "="*70)
    print("🧬 ANTI-CONVERGENCE EVOLUTION")
    print("="*70)
    print(f"Population: {pop_size}")
    print(f"Generations: {n_generations}")
    print(f"Resume from checkpoint: {resume}")
    print("="*70 + "\n")

    start_gen = 0
    population = None
    population_archive = []

    if resume:
        checkpoint = find_latest_checkpoint()
        if checkpoint:
            start_gen = checkpoint['generation']
            population = checkpoint['population']
            population_archive = checkpoint['metadata'].get('archive', [])
            print(f"▶️  Resuming from generation {start_gen}\n")

    if population is None:
        population = toolbox.population(n=pop_size)
        print(f"🆕 Created new population\n")

    for gen in range(start_gen, start_gen + n_generations):
        print(f"\n{'='*70}")
        print(f"📍 GENERATION {gen + 1}")
        print(f"{'='*70}\n")

        for i, ind in enumerate(population):
            if not ind.fitness.valid:
                raw_fitness = evaluate_loss(ind, gen + 1, i + 1)
                novelty_bonus = calculate_novelty_bonus(ind, population_archive)
                trivial_mult = apply_trivial_penalty(ind)
                final_fitness = (raw_fitness + novelty_bonus) * trivial_mult

                ind.fitness.values = (final_fitness,)
                ind.raw_fitness = raw_fitness

                tree_str = str(ind)[:50] + "..." if len(str(ind)) > 50 else str(ind)
                print(f"  ✅ Ind {i+1}: Raw={raw_fitness:.2f}% Final={final_fitness:.2f}% | {tree_str}")

        apply_fitness_sharing(population)
        population.sort(key=lambda x: x.shared_fitness, reverse=True)
        diversity = calculate_population_diversity(population)
        population_archive.extend([toolbox.clone(ind) for ind in population])

        raw_fits = [ind.raw_fitness for ind in population]
        shared_fits = [ind.shared_fitness for ind in population]

        print(f"\n📊 Generation {gen + 1} Statistics:")
        print(f"   Raw Fitness:    Best={max(raw_fits):.2f}% Avg={np.mean(raw_fits):.2f}%")
        print(f"   Shared Fitness: Best={max(shared_fits):.2f}% Avg={np.mean(shared_fits):.2f}%")
        print(f"   Diversity:      {diversity:.3f}")

        metadata = {
            'generation': gen + 1,
            'best_raw_fitness': max(raw_fits),
            'diversity': diversity,
            'archive': population_archive
        }
        save_checkpoint(population, gen + 1, metadata)

        if gen < start_gen + n_generations - 1:
            mut_rate = adaptive_mutation_rate(gen + 1, diversity)
            next_gen = [toolbox.clone(ind) for ind in population[:3]]
            while len(next_gen) < pop_size:
                p1 = max(random.sample(population, 3), key=lambda x: x.shared_fitness)
                p2 = max(random.sample(population, 3), key=lambda x: x.shared_fitness)
                child = toolbox.clone(p1)
                if random.random() < 0.5: gp.cxOnePoint(child, toolbox.clone(p2))
                if random.random() < mut_rate: gp.mutUniform(child, expr=toolbox.expr, pset=pset)
                del child.fitness.values
                next_gen.append(child)
            population = next_gen

    return population

print("✅ Evolution system restored and ready")

✅ Evolution system restored and ready


In [ ]:
# ============================================================================
# RUN THE EVOLUTION
# ============================================================================

final_population = run_anti_convergence_evolution(
    pop_size=20,
    n_generations=20,
    resume=True  # Will resume from checkpoint if available
)


🧬 ANTI-CONVERGENCE EVOLUTION
Population: 20
Generations: 20
Resume from checkpoint: True

📂 Loaded checkpoint: gen 1
▶️  Resuming from generation 1


📍 GENERATION 2


📊 Generation 2 Statistics:
   Raw Fitness:    Best=43.47% Avg=3.85%
   Shared Fitness: Best=43.19% Avg=5.52%
   Diversity:      0.936
💾 Saved checkpoint: gen 2

📍 GENERATION 3

  ✅ Ind 4: Raw=2.00% Final=4.00% | tanh(min(error, 1.9345394946000647))
  ✅ Ind 5: Raw=2.00% Final=4.00% | tanh(sqrt(error))
  ✅ Ind 6: Raw=2.00% Final=4.00% | log(square(error))
  ✅ Ind 7: Raw=2.00% Final=4.00% | add(max(error, error), sqrt(0.4380090778562109))
  ✅ Ind 8: Raw=48.93% Final=46.96% | square(error)
  ✅ Ind 9: Raw=1.47% Final=1.97% | log(abs(log(0.21330653484472184)))
  ✅ Ind 10: Raw=2.00% Final=2.50% | max(softplus(cos(1.169120183304332)), cos(max(erro...
  ✅ Ind 11: Raw=2.67% Final=4.67% | tanh(neg(softplus(0.19835692335808397)))
  ✅ Ind 12: Raw=2.00% Final=4.00% | tanh(sqrt(0.4380090778562109))
  ✅ Ind 13: Raw=2.00% Final=2.50% | s

In [ ]:
# Load the latest checkpoint to access the full history archive
latest_cp = find_latest_checkpoint()

if latest_cp and 'metadata' in latest_cp:
    archive = latest_cp['metadata'].get('archive', [])

    if archive:
        # Find the absolute best by raw fitness
        all_time_best = max(archive, key=lambda x: getattr(x, 'raw_fitness', 0))

        print("="*70)
        print("⭐ ALL-TIME HISTORICAL BEST (RAW FITNESS)")
        print("="*70)
        print(f"Raw Accuracy: {all_time_best.raw_fitness:.2f}%")
        print(f"Expression: {str(all_time_best)}")
        print("="*70)

        # Check if it was trivial or highly similar to others
        print(f"\nIs Trivial: {is_trivial_tree(all_time_best)}")
        structure_hash = get_tree_structure_hash(all_time_best)
        occurrences = sum(1 for x in archive if get_tree_structure_hash(x) == structure_hash)
        print(f"Structural occurrences in history: {occurrences}")
    else:
        print("⚠️ Archive is empty in the checkpoint.")
else:
    print("⚠️ Could not load checkpoint or metadata.")

📂 Loaded checkpoint: gen 21
⭐ ALL-TIME HISTORICAL BEST (RAW FITNESS)
Raw Accuracy: 56.27%
Expression: square(sub(error, 0.7243011650868414))

Is Trivial: False
Structural occurrences in history: 12


In [ ]:
# Continue evolution for 10 more generations
# This will resume from the Gen 21 checkpoint
extended_population = run_anti_convergence_evolution(
    pop_size=20,
    n_generations=10,
    resume=True
)


🧬 ANTI-CONVERGENCE EVOLUTION
Population: 20
Generations: 10
Resume from checkpoint: True

📂 Loaded checkpoint: gen 21
▶️  Resuming from generation 21


📍 GENERATION 22


📊 Generation 22 Statistics:
   Raw Fitness:    Best=50.53% Avg=30.67%
   Shared Fitness: Best=51.87% Avg=22.79%
   Diversity:      0.834
💾 Saved checkpoint: gen 22

📍 GENERATION 23

  ✅ Ind 4: Raw=49.33% Final=49.33% | square(sub(abs(sigmoid(add(add(1.8230308761047436,...
  ✅ Ind 5: Raw=46.67% Final=46.67% | square(sub(sin(mul(tanh(neg(error)), abs(softplus(...
  ✅ Ind 6: Raw=38.80% Final=40.80% | square(sub(mul(tanh(error), abs(softplus(neg(sin(e...
  ✅ Ind 7: Raw=0.93% Final=1.43% | square(0.7243011650868414)
  ✅ Ind 8: Raw=45.47% Final=47.47% | square(sub(sin(softplus(error)), error))
  ✅ Ind 9: Raw=40.93% Final=40.93% | square(sub(sin(mul(tanh(neg(error)), abs(softplus(...
  ✅ Ind 10: Raw=30.00% Final=30.00% | square(sub(abs(sigmoid(error)), sub(error, 0.72430...
  ✅ Ind 11: Raw=45.47% Final=45.47% | square(sub(ab

In [ ]:
# ============================================================================
# FINAL SHOWDOWN: Gen 30 Dynamic Scaled Loss vs CrossEntropy
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class Gen30Loss(nn.Module):
    """
    Evolved Loss Function from Generation 30.
    Formula: (Error * abs(softplus(Error)) - Error)^2

    This function dynamically scales the gradient based on network confidence,
    acting as a learned regularization mechanism to prevent overfitting in
    few-shot scenarios.
    """
    def __init__(self):
        super(Gen30Loss, self).__init__()

    def forward(self, logits, targets):
        # 1. Calculate standard probabilities and error
        probs = F.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs)
        one_hot.scatter_(1, targets.view(-1, 1), 1)
        error = one_hot - probs

        # 2. Apply the Evolved Gen 30 Math
        scaled_error = error * torch.abs(F.softplus(error))
        loss_val = torch.square(scaled_error - error)

        # 3. Aggregate the loss
        final_loss = torch.mean(torch.sum(loss_val, dim=1))

        return final_loss

def train_and_test(loss_fn, loss_name, epochs=25):
    print(f"\n{'='*70}")
    print(f"🔥 Training: {loss_name}")
    print(f"{'='*70}")

    model = OmniglotCNN(num_classes=50).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            if torch.isnan(loss) or torch.isinf(loss):
                print(f"   ⚠️ NaN/Inf loss encountered at epoch {epoch+1}. Halting.")
                return 0.0

            loss.backward()
            optimizer.step()

        if (epoch + 1) % 5 == 0:
            print(f"   Epoch {epoch+1}/{epochs}")

    # Test Phase
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"✅ {loss_name}: {acc:.2f}%\n")
    return acc

print("\n" + "="*70)
print("🏁 THE FINAL EVALUATION")
print("="*70)

# Test the hardcoded Gen 30 Loss
gen30_acc = train_and_test(Gen30Loss(), "Gen 30 Dynamic Scaled Loss", epochs=25)

# Test the PyTorch Baseline
ce_acc = train_and_test(nn.CrossEntropyLoss(), "Standard CrossEntropy", epochs=25)

# Final Results Output
print("="*70)
print("📊 FINAL RESULTS - Omniglot 5-Shot Learning")
print("="*70)
print(f"\nGen 30 Loss:      {gen30_acc:.2f}%")
print(f"CrossEntropy:     {ce_acc:.2f}%")
print(f"Difference:       {gen30_acc - ce_acc:+.2f}%")

if gen30_acc > ce_acc:
    print("\n🎉 EVOLVED LOSS WINS! The dynamic scaling generalizes better than CrossEntropy.")
elif gen30_acc > ce_acc - 1.0:
    print("\n✅ Competitive (within 1%). The evolved loss holds its own.")
else:
    print("\n⚠️ CrossEntropy wins.")

print("="*70)


🏁 THE FINAL EVALUATION

🔥 Training: Gen 30 Dynamic Scaled Loss
   Epoch 5/25
   Epoch 10/25
   Epoch 15/25
   Epoch 20/25
   Epoch 25/25
✅ Gen 30 Dynamic Scaled Loss: 53.60%


🔥 Training: Standard CrossEntropy
   Epoch 5/25
   Epoch 10/25
   Epoch 15/25
   Epoch 20/25
   Epoch 25/25
✅ Standard CrossEntropy: 55.60%

📊 FINAL RESULTS - Omniglot 5-Shot Learning

Gen 30 Loss:      53.60%
CrossEntropy:     55.60%
Difference:       -2.00%

⚠️ CrossEntropy wins.


In [ ]:
# ============================================================================
# FINAL SHOWDOWN: Gen 30 Evolved Loss vs Triplet Margin Loss
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random

# ----------------------------------------------------------------------------
# 1. HARDCODED GEN 30 LOSS
# ----------------------------------------------------------------------------
class Gen30Loss(nn.Module):
    def __init__(self):
        super(Gen30Loss, self).__init__()

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs)
        one_hot.scatter_(1, targets.view(-1, 1), 1)
        error = one_hot - probs

        # The Evolved Gen 30 Math: dynamically scaled gradient
        scaled_error = error * torch.abs(F.softplus(error))
        loss_val = torch.square(scaled_error - error)
        return torch.mean(torch.sum(loss_val, dim=1))

# ----------------------------------------------------------------------------
# 2. DUAL-OUTPUT PROXY MODEL
# ----------------------------------------------------------------------------
class DualOmniglotCNN(nn.Module):
    """Modified to return both the embedding (for Triplet) and logits (for Gen 30)"""
    def __init__(self, num_classes=50):
        super(DualOmniglotCNN, self).__init__()
        # Using the slimmed proxy architecture for rapid evaluation
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(32 * 7 * 7, 128)  # This is our Embedding Layer
        self.fc2 = nn.Linear(128, num_classes) # This is our Classification Layer
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)

        embedding = self.fc1(x) # Extract embedding for Triplet Loss
        x_drop = self.dropout(F.relu(embedding))
        logits = self.fc2(x_drop) # Extract logits for Gen 30

        return embedding, logits

# ----------------------------------------------------------------------------
# 3. TRIPLET DATASET WRAPPER
# ----------------------------------------------------------------------------
class FastTripletWrapper(Dataset):
    """Wraps your existing train_data to generate Anchor, Positive, Negative triplets"""
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.label_to_idx = {}
        # Group indices by class
        for i in range(len(self.base)):
            _, label = self.base[i]
            if label not in self.label_to_idx:
                self.label_to_idx[label] = []
            self.label_to_idx[label].append(i)
        self.labels = list(self.label_to_idx.keys())

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        anchor_img, label = self.base[idx]

        # Pick a positive (same class, different image if possible)
        pos_idx = random.choice(self.label_to_idx[label])
        pos_img, _ = self.base[pos_idx]

        # Pick a negative (completely different class)
        neg_label = random.choice([l for l in self.labels if l != label])
        neg_idx = random.choice(self.label_to_idx[neg_label])
        neg_img, _ = self.base[neg_idx]

        return anchor_img, pos_img, neg_img, label

# Initialize Triplet Loader
triplet_train_loader = DataLoader(FastTripletWrapper(train_data), batch_size=32, shuffle=True, num_workers=0)

# ----------------------------------------------------------------------------
# 4. STANDARD TRAINING LOOP (For Gen 30)
# ----------------------------------------------------------------------------
def train_and_test_standard(loss_fn, loss_name, epochs=25):
    print(f"\n{'='*70}")
    print(f"🔥 Training: {loss_name}")
    print(f"{'='*70}")

    model = DualOmniglotCNN(num_classes=50).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # Train
    model.train()
    for epoch in range(epochs):
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            _, logits = model(inputs) # Ignore embeddings, use logits
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 5 == 0: print(f"   Epoch {epoch+1}/{epochs}")

    # Test
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, logits = model(inputs)
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"✅ {loss_name}: {acc:.2f}%\n")
    return acc

# ----------------------------------------------------------------------------
# 5. METRIC LEARNING TRAINING LOOP (For Triplet Loss)
# ----------------------------------------------------------------------------
def train_and_test_triplet(epochs=25):
    print(f"\n{'='*70}")
    print(f"🔥 Training: Triplet Margin Loss")
    print(f"{'='*70}")

    model = DualOmniglotCNN(num_classes=50).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)

    # Train (Metric Space)
    model.train()
    for epoch in range(epochs):
        for anchor, positive, negative, _ in triplet_train_loader:
            anchor, positive, negative = anchor.to(DEVICE), positive.to(DEVICE), negative.to(DEVICE)
            optimizer.zero_grad()

            # Extract embeddings
            emb_a, _ = model(anchor)
            emb_p, _ = model(positive)
            emb_n, _ = model(negative)

            loss = triplet_loss(emb_a, emb_p, emb_n)
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 5 == 0: print(f"   Epoch {epoch+1}/{epochs}")

    # Test (Prototypical / Nearest Centroid Evaluation)
    model.eval()
    print("   Building class prototypes for evaluation...")
    prototypes = torch.zeros((50, 128)).to(DEVICE)
    counts = torch.zeros(50).to(DEVICE)

    with torch.no_grad():
        # Build Support Set Embeddings
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            embs, _ = model(inputs)
            for i in range(len(labels)):
                prototypes[labels[i]] += embs[i]
                counts[labels[i]] += 1
        prototypes = prototypes / counts.unsqueeze(1) # Average vectors

        # Test against Query Set
        correct, total = 0, 0
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            embs, _ = model(inputs)

            # Find closest prototype via Euclidean Distance
            distances = torch.cdist(embs, prototypes)
            predicted = torch.argmin(distances, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    print(f"✅ Triplet Margin Loss: {acc:.2f}%\n")
    return acc

# ----------------------------------------------------------------------------
# 6. EXECUTE THE SHOWDOWN
# ----------------------------------------------------------------------------
print("\n" + "="*70)
print("🏁 THE ULTIMATE SHOWDOWN: Gen 30 vs Triplet vs CrossEntropy")
print("="*70)

EPOCHS = 25

# 1. Gen 30 Evolved Loss (Classification)
gen30_acc = train_and_test_standard(Gen30Loss(), "Gen 30 Dynamic Scaled Loss", epochs=EPOCHS)

# 2. Standard CrossEntropy (Baseline)
ce_acc = train_and_test_standard(nn.CrossEntropyLoss(), "Standard CrossEntropy", epochs=EPOCHS)

# 3. Triplet Margin Loss (Metric Learning)
triplet_acc = train_and_test_triplet(epochs=EPOCHS)

# Results
print("="*70)
print("📊 FINAL RESULTS - Omniglot 5-Shot Learning")
print("="*70)
print(f"1. Gen 30 Loss:     {gen30_acc:.2f}%")
print(f"2. Triplet Loss:    {triplet_acc:.2f}%")
print(f"3. CrossEntropy:    {ce_acc:.2f}%")
print("="*70)

best_acc = max(gen30_acc, ce_acc, triplet_acc)
if best_acc == gen30_acc:
    print("🏆 WINNER: EVOLVED GEN 30 LOSS!")
elif best_acc == triplet_acc:
    print("🏆 WINNER: TRIPLET MARGIN LOSS!")
else:
    print("🏆 WINNER: CROSS ENTROPY!")


🏁 THE ULTIMATE SHOWDOWN: Gen 30 vs Triplet vs CrossEntropy

🔥 Training: Gen 30 Dynamic Scaled Loss
   Epoch 5/25
   Epoch 10/25
   Epoch 15/25
   Epoch 20/25
   Epoch 25/25
✅ Gen 30 Dynamic Scaled Loss: 57.47%


🔥 Training: Standard CrossEntropy
   Epoch 5/25
   Epoch 10/25
   Epoch 15/25
   Epoch 20/25
   Epoch 25/25
✅ Standard CrossEntropy: 53.73%


🔥 Training: Triplet Margin Loss
   Epoch 5/25
   Epoch 10/25
   Epoch 15/25
   Epoch 20/25
   Epoch 25/25
   Building class prototypes for evaluation...
✅ Triplet Margin Loss: 54.80%

📊 FINAL RESULTS - Omniglot 5-Shot Learning
1. Gen 30 Loss:     57.47%
2. Triplet Loss:    54.80%
3. CrossEntropy:    53.73%
🏆 WINNER: EVOLVED GEN 30 LOSS!


In [ ]:
# ============================================================================
# K-FOLD VALIDATION: De-Lucking the Gen 30 Evolved Loss
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np

# ----------------------------------------------------------------------------
# 1. CORE CLASSES (Gen 30 Loss, Dual Model, Triplet Wrapper, Fast Dataset)
# ----------------------------------------------------------------------------
class Gen30Loss(nn.Module):
    def __init__(self):
        super(Gen30Loss, self).__init__()
    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        one_hot = torch.zeros_like(probs)
        one_hot.scatter_(1, targets.view(-1, 1), 1)
        error = one_hot - probs
        scaled_error = error * torch.abs(F.softplus(error))
        loss_val = torch.square(scaled_error - error)
        return torch.mean(torch.sum(loss_val, dim=1))

class DualOmniglotCNN(nn.Module):
    def __init__(self, num_classes=50):
        super(DualOmniglotCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        embedding = self.fc1(x)
        x_drop = self.dropout(F.relu(embedding))
        logits = self.fc2(x_drop)
        return embedding, logits

class FastFewShotOmniglot(Dataset):
    def __init__(self, base_dataset, shots=5, ways=50, for_train=True, seed=42):
        random.seed(seed)
        class_to_idx = {}
        for i, (_, label) in enumerate(base_dataset._flat_character_images):
            if label not in class_to_idx:
                class_to_idx[label] = []
            class_to_idx[label].append(i)

        selected = random.sample(list(class_to_idx.keys()), ways)
        self.label_map = {old: new for new, old in enumerate(selected)}

        self.indices = []
        for old_label in selected:
            all_idx = class_to_idx[old_label]
            shuffled = random.sample(all_idx, len(all_idx))
            if for_train:
                self.indices.extend(shuffled[:shots])
            else:
                self.indices.extend(shuffled[shots:shots+15])
        self.base = base_dataset

    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        img, old_label = self.base[self.indices[idx]]
        return img, self.label_map[old_label]

class FastTripletWrapper(Dataset):
    def __init__(self, base_dataset):
        self.base = base_dataset
        self.label_to_idx = {}
        for i in range(len(self.base)):
            _, label = self.base[i]
            if label not in self.label_to_idx:
                self.label_to_idx[label] = []
            self.label_to_idx[label].append(i)
        self.labels = list(self.label_to_idx.keys())

    def __len__(self): return len(self.base)
    def __getitem__(self, idx):
        anchor_img, label = self.base[idx]
        pos_idx = random.choice(self.label_to_idx[label])
        pos_img, _ = self.base[pos_idx]
        neg_label = random.choice([l for l in self.labels if l != label])
        neg_idx = random.choice(self.label_to_idx[neg_label])
        neg_img, _ = self.base[neg_idx]
        return anchor_img, pos_img, neg_img, label

# ----------------------------------------------------------------------------
# 2. DYNAMIC DATALOADER GENERATOR
# ----------------------------------------------------------------------------
def generate_seed_loaders(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_data = FastFewShotOmniglot(omniglot, shots=5, ways=50, for_train=True, seed=seed)
    test_data = FastFewShotOmniglot(omniglot, shots=5, ways=50, for_train=False, seed=seed)

    tl = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=0)
    vl = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=0)
    trip_l = DataLoader(FastTripletWrapper(train_data), batch_size=32, shuffle=True, num_workers=0)

    return tl, vl, trip_l

# ----------------------------------------------------------------------------
# 3. TRAINING EVALUATORS
# ----------------------------------------------------------------------------
def run_standard(loss_fn, t_loader, v_loader, epochs=25):
    model = DualOmniglotCNN(num_classes=50).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    model.train()
    for epoch in range(epochs):
        for inputs, labels in t_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            _, logits = model(inputs)
            loss = loss_fn(logits, labels)
            loss.backward()
            optimizer.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in v_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, logits = model(inputs)
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def run_triplet(t_loader, v_loader, trip_loader, epochs=25):
    model = DualOmniglotCNN(num_classes=50).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    triplet_loss = nn.TripletMarginLoss(margin=1.0, p=2)

    model.train()
    for epoch in range(epochs):
        for anchor, positive, negative, _ in trip_loader:
            anchor, positive, negative = anchor.to(DEVICE), positive.to(DEVICE), negative.to(DEVICE)
            optimizer.zero_grad()
            emb_a, _ = model(anchor)
            emb_p, _ = model(positive)
            emb_n, _ = model(negative)
            loss = triplet_loss(emb_a, emb_p, emb_n)
            loss.backward()
            optimizer.step()

    model.eval()
    prototypes = torch.zeros((50, 128)).to(DEVICE)
    counts = torch.zeros(50).to(DEVICE)
    with torch.no_grad():
        for inputs, labels in t_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            embs, _ = model(inputs)
            for i in range(len(labels)):
                prototypes[labels[i]] += embs[i]
                counts[labels[i]] += 1
        prototypes = prototypes / counts.unsqueeze(1)

        correct, total = 0, 0
        for inputs, labels in v_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            embs, _ = model(inputs)
            distances = torch.cdist(embs, prototypes)
            predicted = torch.argmin(distances, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

# ----------------------------------------------------------------------------
# 4. EXECUTE K-FOLD VERIFICATION
# ----------------------------------------------------------------------------
print("\n" + "="*70)
print("🏁 COMMENCING K-FOLD VERIFICATION (5 Different Random Seeds)")
print("="*70)

SEEDS = [42, 100, 2026, 777, 999]
EPOCHS = 25

results = {
    "Gen30": [],
    "CrossEntropy": [],
    "Triplet": []
}

for i, seed in enumerate(SEEDS):
    print(f"\n🌀 Running Fold {i+1}/5 (Seed: {seed})")
    print("-" * 50)

    t_loader, v_loader, trip_loader = generate_seed_loaders(seed)

    acc_gen30 = run_standard(Gen30Loss(), t_loader, v_loader, epochs=EPOCHS)
    print(f"  ✅ Gen 30 Evolved:     {acc_gen30:.2f}%")

    acc_ce = run_standard(nn.CrossEntropyLoss(), t_loader, v_loader, epochs=EPOCHS)
    print(f"  ✅ CrossEntropy:       {acc_ce:.2f}%")

    acc_trip = run_triplet(t_loader, v_loader, trip_loader, epochs=EPOCHS)
    print(f"  ✅ Triplet Margin:     {acc_trip:.2f}%")

    results["Gen30"].append(acc_gen30)
    results["CrossEntropy"].append(acc_ce)
    results["Triplet"].append(acc_trip)

# ----------------------------------------------------------------------------
# 5. THE FINAL VERDICT
# ----------------------------------------------------------------------------
print("\n" + "="*70)
print("🏆 THE FINAL K-FOLD VERDICT 🏆")
print("="*70)

avg_gen30 = np.mean(results["Gen30"])
std_gen30 = np.std(results["Gen30"])

avg_ce = np.mean(results["CrossEntropy"])
std_ce = np.std(results["CrossEntropy"])

avg_trip = np.mean(results["Triplet"])
std_trip = np.std(results["Triplet"])

print(f"Gen 30 Evolved Loss:    {avg_gen30:.2f}% (±{std_gen30:.2f}%)")
print(f"Triplet Margin Loss:    {avg_trip:.2f}% (±{std_trip:.2f}%)")
print(f"Standard CrossEntropy:  {avg_ce:.2f}% (±{std_ce:.2f}%)")
print("="*70)

best_avg = max(avg_gen30, avg_ce, avg_trip)

if best_avg == avg_gen30:
    print("\n🎉 MATHEMATICALLY PROVEN: Gen 30 is the superior loss function for this architecture!")
    if avg_gen30 > avg_trip + 1.0:
        print("💡 It consistently beat Triplet Loss across multiple dataset splits. You have a publishable result.")
elif best_avg == avg_trip:
    print("\n⚖️ Reality Check: Triplet Loss won the average. Gen 30 is a powerful approximation, but PyTorch metric learning is stronger.")
else:
    print("\n⚖️ Reality Check: CrossEntropy won the average. The GP heavily overfit to the first dataset layout.")


🏁 COMMENCING K-FOLD VERIFICATION (5 Different Random Seeds)

🌀 Running Fold 1/5 (Seed: 42)
--------------------------------------------------
  ✅ Gen 30 Evolved:     54.53%
  ✅ CrossEntropy:       51.47%
  ✅ Triplet Margin:     54.53%

🌀 Running Fold 2/5 (Seed: 100)
--------------------------------------------------
  ✅ Gen 30 Evolved:     53.20%
  ✅ CrossEntropy:       65.47%
  ✅ Triplet Margin:     57.73%

🌀 Running Fold 3/5 (Seed: 2026)
--------------------------------------------------
  ✅ Gen 30 Evolved:     59.33%
  ✅ CrossEntropy:       60.13%
  ✅ Triplet Margin:     63.47%

🌀 Running Fold 4/5 (Seed: 777)
--------------------------------------------------
  ✅ Gen 30 Evolved:     55.47%
  ✅ CrossEntropy:       64.00%
  ✅ Triplet Margin:     54.00%

🌀 Running Fold 5/5 (Seed: 999)
--------------------------------------------------
  ✅ Gen 30 Evolved:     59.20%
  ✅ CrossEntropy:       60.13%
  ✅ Triplet Margin:     60.00%

🏆 THE FINAL K-FOLD VERDICT 🏆
Gen 30 Evolved Loss:    56.35